# Czas w C++: `std::chrono`

Laboratorium o mierzeniu i reprezentowaniu czasu w C++ z biblioteką `<chrono>`: odcinki czasu (*durations*),
zegary i punkty w czasie, kalendarz, testowanie kodu zależnego od czasu, a na koniec strefy czasowe
i sekundy przestępne.

## Cele

Po tym laboratorium potrafisz:

* zapisać czas w typach, które pilnują jednostek (`std::chrono::milliseconds` zamiast `int`), i świadomie wybrać
  sposób konwersji: niejawny, `duration_cast`, `floor`, `ceil` albo `round`,
* dobrać zegar do zadania: `system_clock` do daty i godziny, `steady_clock` do mierzenia i timeoutów,
* liczyć na datach (`year_month_day`, `sys_days`) bez pułapek końca miesiąca i lat przestępnych,
* napisać klasę zależną od czasu tak, by dało się ją przetestować bez czekania — ze sztucznym zegarem.

## Dlaczego czas jest trudny?

Czas wygląda na zwykłą liczbę — i właśnie dlatego łatwo o błędy, które ujawniają się rzadko, za to boleśnie:

* **Zegar, który cofa się w czasie.** W noc sylwestrową 2016/2017 do doby dodano sekundę przestępną. Kod DNS w Cloudflare
  liczył czas trwania jako różnicę dwóch odczytów zegara systemowego i dostał liczbę ujemną.
  Przyczyna według raportu firmy: *„the belief that time cannot go backwards”*
  ([Cloudflare, 2017](https://blog.cloudflare.com/how-and-why-the-leap-second-affected-cloudflare-dns/)).
  Właśnie do takich pomiarów służy `steady_clock` — o nim w notebooku 02.
* **Data, która nie istnieje.** 29 lutego 2012 Windows Azure wystawiał certyfikaty ważne „rok od dziś”,
  dodając jedynkę do numeru roku. Wyszło 29 lutego 2013 — data, której nie ma, i awaria usługi
  ([Microsoft, 2012](https://azure.microsoft.com/en-us/blog/summary-of-windows-azure-service-disruption-on-feb-29th-2012/)).
  Arytmetyka na datach — notebook 03.

Biblioteka `<chrono>` nie usuwa tych problemów magicznie, ale zamienia wiele z nich w **błędy kompilacji**.
Jej autor, Howard Hinnant, podsumowuje to tak: *„Compile-time errors are favored over run-time errors”*
([slajdy CppCon 2016](https://github.com/CppCon/CppCon2016/raw/master/Tutorials/A%20chrono%20Tutorial/A%20chrono%20Tutorial%20-%20Howard%20Hinnant%20-%20CppCon%202016.pdf), s. 104).

## Trzy zasady na całe laboratorium

1. **Jednostka jest częścią typu.** `seconds`, `milliseconds`, `days` to różne typy; kompilator pilnuje,
   żeby nie pomylić sekund z milisekundami i nie zgubić cicho ułamka.
2. **Punkt w czasie to nie odcinek czasu.** `time_point` (np. „teraz”) i `duration` (np. „5 sekund”)
   mają różne typy i różną arytmetykę — tak jak wskaźnik i liczba całkowita.
3. **Kod nie powinien sam „wiedzieć”, która jest godzina.** Zegar przekazujemy z zewnątrz,
   dzięki czemu w testach można go podmienić i przewijać czas o godzinę w mikrosekundę.

## Jak korzystać z notebooków

* Komórkę uruchamiasz skrótem `Shift+Enter` (albo przyciskiem ▶ na pasku narzędzi).
* Komórki z kodem zaczynające się od `!` wykonują polecenia powłoki — w tym laboratorium
  głównie `cmake` i `ctest`. Każda taka komórka jest niezależna: zaczyna w katalogu notebooka.
* Pliki do edycji otwierasz kliknięciem w link w treści ćwiczenia (otworzą się w edytorze JupyterLab).
  **Zapisz plik** (`Ctrl+S`) przed uruchomieniem komórki, która go kompiluje.
* Przykłady kodu w notebookach są kompletnymi programami — możesz je skopiować do pliku albo
  do [Compiler Explorer](https://godbolt.org/) i uruchomić.

## Wymagania

| Narzędzie | Minimalna wersja |
|---|---|
| Kompilator C++20 | GCC 11.2, Clang 14 (z libstdc++ albo libc++), Visual Studio 2022 17.0 (MSVC 19.30) |
| CMake | 3.22 |
| Python + JupyterLab | Python 3.10, JupyterLab 4 |
| Internet | tylko przy pierwszej kompilacji — CMake pobiera GoogleTest 1.18.0 (raz, do `lab/_deps`) |

GoogleTest 1.18 wymaga C++17 i co najmniej tych wersji kompilatorów i CMake
([polityka wsparcia Google](https://github.com/google/oss-policies-info/blob/main/foundational-cxx-support-matrix.md)).
Domyślne pakiety w Ubuntu 22.04 i 24.04 je spełniają. Na macOS (Apple Clang) laboratorium nie było sprawdzane.

Wszystkie ćwiczenia i przykłady w notebookach 01–04 działają na tych minimalnych wersjach.
Notebook 05 (formatowanie dat, strefy czasowe) pokazuje części C++20, które pojawiły się w kompilatorach później
— wymaga **GCC 13+** albo **Visual Studio 2022 17.2+**; to materiał dla ciekawych, bez ćwiczeń.

## Sprawdzenie środowiska

Poniższe komórki budują mały projekt testowy z katalogu [`_smoke/`](_smoke/smoke_test.cpp).
Używa on tych samych ustawień co wszystkie ćwiczenia (plik [`cmake/lab.cmake`](cmake/lab.cmake)):
standard C++20, GoogleTest i gMock pobrane przez `FetchContent` w przypiętej wersji.

Każde ćwiczenie budujemy tymi samymi trzema krokami:

1. `cmake -S <źródła> -B <katalog build>` — konfiguracja (za pierwszym razem pobiera i konfiguruje GoogleTest),
2. `cmake --build <katalog build>` — kompilacja,
3. `ctest --test-dir <katalog build>` — uruchomienie wszystkich testów.

Opcje `--config Debug` i `-C Debug` są potrzebne tylko w Visual Studio (generator wielokonfiguracyjny);
na Linuksie i macOS nic nie zmieniają.

In [ ]:
# Tool versions
!cmake --version

In [ ]:
# Configure, build and run the smoke test
!cmake -S _smoke -B _smoke/build && cmake --build _smoke/build --config Debug --parallel && ctest --test-dir _smoke/build -C Debug --output-on-failure

Na końcu powinieneś zobaczyć podsumowanie `ctest` z informacją, że przeszły wszystkie testy, np.:

```
100% tests passed, 0 tests failed out of 2
```

(nowsze wersje CMake skracają je do `100% tests passed out of 2`).

Jeśli konfiguracja się nie powiedzie, sprawdź wersję CMake i kompilatora. Komunikat o braku
`<compare>` albo błąd przy `operator<=>` oznacza zbyt stary kompilator.

## Plan laboratorium

| # | Notebook | Temat | Ćwiczenie |
|---|---|---|---|
| 01 | Odcinki czasu | `duration`, literały, konwersje i zaokrąglanie, przepełnienie | Ćw. 1 |
| 02 | Zegary i punkty w czasie | `time_point`, `system_clock` i `steady_clock`, mierzenie, uśpienie, timeouty | Ćw. 2 |
| 03 | Kalendarz | `year_month_day`, `sys_days`, dni tygodnia, ostatni dzień miesiąca, `hh_mm_ss` | Ćw. 3 |
| 04 | Testowanie kodu zależnego od czasu | interfejsy z typami czasu, wstrzykiwanie zegara, `FakeClock` | Ćw. 4 i 5 |
| 05 | Dla ciekawych | `std::format`, strefy czasowe i zmiana czasu, sekundy przestępne, rok 2038, głośne awarie | — |

Wszystkie ćwiczenia opowiadają jedną luźną historię: piszemy fragmenty małego **klienta zewnętrznego API** —
rozliczanie wywołań, timeouty i ponowienia, ważność klucza API, cache odpowiedzi i ogranicznik liczby zapytań.
Każde ćwiczenie da się jednak zrobić niezależnie od pozostałych.

Na start polecamy wykład autora biblioteki:
[Howard Hinnant, „A \<chrono\> Tutorial”, CppCon 2016](https://www.youtube.com/watch?v=P32hvk8b13M)
oraz przegląd na [cppreference](https://en.cppreference.com/w/cpp/chrono.html).